In [ ]:
import pandas as pd
data=pd.read_csv("subdata_pandas/student.csv")

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# ==========================================
# 1. LOAD DATASET
# ==========================================

file_path = Path("subdata_pandas/student.csv")

df = pd.read_csv(file_path)

print("\n========== STUDENT DATASET ==========")
print(df.head())

print("\n========== DATASET INFORMATION ==========")
print(df.info())

print("\n========== MISSING VALUES ==========")
print(df.isnull().sum())


# ==========================================
# 2. DATA CLEANING
# ==========================================

df.columns = df.columns.str.strip()
df.drop_duplicates(inplace=True)

print("\n========== CLEANED DATA ==========")
print("Total records:", len(df))


# ==========================================
# 3. CHECK REQUIRED COLUMNS
# ==========================================

print("\nAvailable columns:")
print(df.columns.tolist())

required_columns = [
    "id", "firstName", "lastName", "age", "marks", "fees"
]

missing_columns = [
    col for col in required_columns if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns: {missing_columns}. "
        "Check the column names in student.csv."
    )


# ==========================================
# 4. PREPARE STUDENT DETAILS
# ==========================================

df["marks"] = pd.to_numeric(df["marks"], errors="coerce")
df["age"] = pd.to_numeric(df["age"], errors="coerce")
df["fees"] = pd.to_numeric(df["fees"], errors="coerce")

df["fullName"] = (
    df["firstName"].fillna("").astype(str)
    + " "
    + df["lastName"].fillna("").astype(str)
).str.strip()

print("\n========== STUDENT DETAILS ==========")
print(df[["id", "fullName", "age", "marks", "fees"]])


# ==========================================
# 5. STUDENT PERFORMANCE ANALYSIS
# ==========================================

print("\n========== PERFORMANCE STATISTICS ==========")
print("Total students:", len(df))
print("Average marks:", round(df["marks"].mean(), 2))
print("Highest marks:", df["marks"].max())
print("Lowest marks:", df["marks"].min())
print("Average age:", round(df["age"].mean(), 2))


# ==========================================
# 6. GRADE CALCULATION
# ==========================================

def calculate_grade(marks):
    if pd.isna(marks):
        return "Not Available"
    elif marks >= 90:
        return "A"
    elif marks >= 80:
        return "B"
    elif marks >= 70:
        return "C"
    elif marks >= 60:
        return "D"
    else:
        return "F"


df["grade"] = df["marks"].apply(calculate_grade)

print("\n========== STUDENT GRADES ==========")
print(df[["fullName", "marks", "grade"]])


# ==========================================
# 7. ATTENDANCE ANALYTICS
# ==========================================
# If attendance columns exist, calculate actual
# attendance percentage. Otherwise, report that
# attendance data is unavailable.

attendance_columns = ["classes_attended", "total_classes"]

if all(col in df.columns for col in attendance_columns):

    df["classes_attended"] = pd.to_numeric(
        df["classes_attended"], errors="coerce"
    )
    df["total_classes"] = pd.to_numeric(
        df["total_classes"], errors="coerce"
    )

    valid = (
        df["total_classes"].notna()
        & (df["total_classes"] > 0)
        & df["classes_attended"].notna()
        & (df["classes_attended"] >= 0)
        & (df["classes_attended"] <= df["total_classes"])
    )

    df["attendance_percentage"] = float("nan")

    df.loc[valid, "attendance_percentage"] = (
        df.loc[valid, "classes_attended"]
        / df.loc[valid, "total_classes"]
        * 100
    ).round(2)

    df["attendance_status"] = df["attendance_percentage"].apply(
        lambda x: (
            "Low Attendance" if pd.notna(x) and x < 75
            else "Satisfactory Attendance" if pd.notna(x)
            else "Data Unavailable"
        )
    )

    print("\n========== ATTENDANCE ANALYSIS ==========")
    print(
        df[
            ["fullName", "attendance_percentage", "attendance_status"]
        ]
    )

    print(
        "Average attendance:",
        round(df["attendance_percentage"].mean(), 2), "%"
    )

    print("\nStudents below 75% attendance:")
    print(
        df.loc[
            df["attendance_percentage"] < 75,
            ["fullName", "attendance_percentage"]
        ]
    )

else:
    print("\n========== ATTENDANCE ANALYSIS ==========")
    print("Attendance data is not available in this dataset.")
    print("Required columns: classes_attended, total_classes")


# ==========================================
# 8. STUDENT RANKING
# ==========================================

ranked_students = df.sort_values(
    by="marks", ascending=False, na_position="last"
).copy()

ranked_students["rank"] = (
    ranked_students["marks"]
    .rank(method="min", ascending=False)
)

print("\n========== STUDENT RANKING ==========")
print(ranked_students[["fullName", "marks", "rank"]])


# ==========================================
# 9. GRADE DISTRIBUTION
# ==========================================

print("\n========== GRADE DISTRIBUTION ==========")
print(df["grade"].value_counts())


# ==========================================
# 10. VISUALIZE STUDENT MARKS
# ==========================================

plot_data = df.dropna(subset=["marks"])

if not plot_data.empty:
    plt.figure(figsize=(10, 5))
    sns.barplot(data=plot_data, x="fullName", y="marks")
    plt.title("Student Marks Analysis")
    plt.xlabel("Student Name")
    plt.ylabel("Marks")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


# ==========================================
# 11. VISUALIZE GRADE DISTRIBUTION
# ==========================================

grade_counts = df.loc[
    df["grade"] != "Not Available", "grade"
].value_counts()

if not grade_counts.empty:
    plt.figure(figsize=(7, 7))
    grade_counts.plot(kind="pie", autopct="%1.1f%%")
    plt.title("Student Grade Distribution")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()


# ==========================================
# 12. ATTENDANCE CHART, IF AVAILABLE
# ==========================================

if "attendance_percentage" in df.columns:
    attendance_data = df.dropna(
        subset=["attendance_percentage"]
    )

    if not attendance_data.empty:
        plt.figure(figsize=(10, 5))
        sns.barplot(
            data=attendance_data,
            x="fullName",
            y="attendance_percentage"
        )
        plt.axhline(
            75, linestyle="--", label="75% threshold"
        )
        plt.title("Student Attendance Analytics")
        plt.xlabel("Student Name")
        plt.ylabel("Attendance Percentage (%)")
        plt.xticks(rotation=45, ha="right")
        plt.legend()
        plt.tight_layout()
        plt.show()


# ==========================================
# 13. SAVE ANALYSIS RESULTS
# ==========================================

output_path = Path("subdata_pandas/student_analysis_results.csv")

df.to_csv(output_path, index=False)

print("\n========== COMPLETED ==========")
print("Analysis results saved to:", output_path)


: 